# Warble: fine-tune Whisper on your voice

**Setup (one time):**
1. Open [Google Drive](https://drive.google.com) in your browser and drag your local `warble` folder into `My Drive` — this uploads the code AND your `data/raw` recordings.
2. Upload this notebook to [colab.research.google.com](https://colab.research.google.com) (File → Upload notebook).
3. Runtime → Change runtime type → **T4 GPU**.

Then run the cells top to bottom. Training runs on Colab's fast local disk; the **last cell copies the finished model into Drive and downloads a zip straight to your browser** — no Drive desktop app needed. To use the model locally afterwards:

```bash
unzip ~/Downloads/whisper-warble-final.zip -d models/whisper-warble/
```

**Kaggle alternative:** upload the `warble` folder as a Kaggle Dataset, attach it to a GPU notebook (P100), `cd` into it, and run the same commands (skip the `google.colab` cells and grab the model from the notebook's output files).

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Adjust this path if you put the folder somewhere else in Drive
%cd "/content/drive/MyDrive/warble"
!ls

In [ ]:
!pip install -q -r requirements.txt

## 1. Validate clips and build train/val/test splits

In [ ]:
!python training/prepare_dataset.py

## 2. Fine-tune whisper-small

Trains to `/content` (Colab's local disk — much faster and more reliable than writing checkpoints into mounted Drive). The finished model is copied to Drive in the last cell.

Defaults are tuned for a free T4 (16 GB). If you see out-of-memory errors, drop `--batch-size` to 4 and raise `--grad-accum` to 8. If training overfits badly (val WER rises while train WER falls), re-run with `--freeze-encoder`. If the runtime disconnects mid-training, just re-run this cell.

In [ ]:
!python training/train.py \
    --model openai/whisper-small \
    --epochs 3 \
    --batch-size 8 \
    --grad-accum 4 \
    --fp16 \
    --num-workers 2 \
    --output-dir /content/whisper-warble

## 3. Evaluate: stock vs fine-tuned on held-out test clips

In [ ]:
!python training/eval.py \
    --model-dir /content/whisper-warble/final \
    --base-model openai/whisper-small \
    --report /content/whisper-warble/eval_results.json

## 4. Bring the model home

Copies the final model into your Drive folder (so it survives even if this runtime shuts down) and downloads it as a zip through your browser (~1 GB). Back on your Mac:

```bash
unzip ~/Downloads/whisper-warble-final.zip -d models/whisper-warble/
```

In [ ]:
import os, shutil
from google.colab import files

# Keep a copy in Drive (we're cd'd into the warble repo there)
os.makedirs("models/whisper-warble", exist_ok=True)
shutil.copytree("/content/whisper-warble/final", "models/whisper-warble/final", dirs_exist_ok=True)
if os.path.exists("/content/whisper-warble/eval_results.json"):
    shutil.copy("/content/whisper-warble/eval_results.json", "models/eval_results.json")

# Direct browser download
shutil.make_archive("/content/whisper-warble-final", "zip", "/content/whisper-warble", "final")
files.download("/content/whisper-warble-final.zip")

## Done — using the model on your Mac

After unzipping into `models/whisper-warble/`:

```bash
.venv/bin/python training/transcribe.py some_audio.m4a
```

Next up: Phase 4 — serving it from a local FastAPI backend to the browser.